# LeetCode 181 - Employees Earning More Than Their Managers (Practice Harness)

Run all cells after editing only `SOLUTION_SQL`.

Rules this harness enforces:
- Output must have exactly one column named `Employee`.
- Result must match expected output for every test case.
- Row order does not matter (`any order` is accepted).


In [ ]:
import duckdb

def run_case(solution_sql: str, rows: list[tuple[int, str, int, int | None]], case_name: str):
    con = duckdb.connect(':memory:')
    con.execute('CREATE TABLE Employee (id INT, name VARCHAR, salary INT, managerId INT);')

    if rows:
        con.executemany('INSERT INTO Employee (id, name, salary, managerId) VALUES (?, ?, ?, ?);', rows)

    actual = con.execute(solution_sql).fetchall()
    cols = [d[0] for d in con.description]

    if cols != ['Employee']:
        raise AssertionError(
            f"[{case_name}] Expected one column named 'Employee', got {cols}"
        )

    normalized = sorted([r[0] for r in actual])
    return normalized


In [ ]:
# Test fixtures: (case_name, employee_rows, expected_employees)
TEST_CASES = [
    (
        'example_case',
        [
            (1, 'Joe', 70000, 3),
            (2, 'Henry', 80000, 4),
            (3, 'Sam', 60000, None),
            (4, 'Max', 90000, None),
        ],
        ['Joe'],
    ),
    (
        'multiple_employees_match',
        [
            (1, 'Ava', 100, None),
            (2, 'Ben', 200, 1),
            (3, 'Cara', 300, 1),
            (4, 'Dan', 50, 2),
        ],
        ['Ben', 'Cara'],
    ),
    (
        'no_one_earns_more',
        [
            (1, 'Boss', 1000, None),
            (2, 'E1', 900, 1),
            (3, 'E2', 800, 1),
        ],
        [],
    ),
    (
        'duplicate_names_are_allowed',
        [
            (1, 'Mgr1', 100, None),
            (2, 'Mgr2', 100, None),
            (3, 'Alex', 150, 1),
            (4, 'Alex', 180, 2),
        ],
        ['Alex', 'Alex'],
    ),
]


In [ ]:
# ========================= EDIT ONLY THIS STRING =========================
SOLUTION_SQL = """
    SELECT e.name AS Employee FROM Employee e WHERE e.salary > (SELECT u.salary FROM Employee u WHERE u.id = e.managerId)
"""
# ======================================================================


In [ ]:
results = []

for case_name, rows, expected in TEST_CASES:
    actual = run_case(SOLUTION_SQL, rows, case_name)
    passed = (actual == sorted(expected))
    results.append((case_name, sorted(expected), actual, passed))

print(f"{'case':30} {'expected_count':>14} {'actual_count':>12} {'pass':>6}")
print('-' * 70)
for case_name, expected, actual, passed in results:
    print(f"{case_name:30} {len(expected):>14} {len(actual):>12} {str(passed):>6}")

all_passed = all(r[3] for r in results)
print('\nOVERALL:', 'PASS' if all_passed else 'FAIL')

if not all_passed:
    print('\nMISMATCH DETAILS:')
    for case_name, expected, actual, passed in results:
        if not passed:
            print(f"\n[{case_name}] expected={expected}")
            print(f"[{case_name}] actual  ={actual}")
    failed = [r[0] for r in results if not r[3]]
    raise AssertionError(f'Failed cases: {failed}')


## EBNF Hint: Correlated Subquery Pattern
```ebnf
query                = "SELECT", select_list, "FROM", table, outer_alias,
                       "WHERE", outer_col, compare_op, "(", correlated_subquery, ")" ;

correlated_subquery  = "SELECT", scalar_expr,
                       "FROM", same_or_other_table, inner_alias,
                       "WHERE", correlation_predicate ;

correlation_predicate = inner_fk_col, "=", outer_ref_col ;

select_list          = identifier | identifier, {",", identifier} ;
scalar_expr          = identifier | aggregate_expr ;
aggregate_expr       = func_name, "(", identifier, ")" ;
compare_op           = ">" | "<" | ">=" | "<=" | "=" | "<>" ;

table                = identifier ;
same_or_other_table  = identifier ;
outer_alias          = identifier ;
inner_alias          = identifier ;
outer_col            = identifier ;
inner_fk_col         = identifier ;
outer_ref_col        = identifier ;
identifier           = letter, {letter | digit | "_"} ;
```

Key idea: `outer_ref_col` comes from the outer row alias, so the subquery is re-evaluated per outer row.


## Usage
1. Change only `SOLUTION_SQL`.
2. Run all cells.
3. Fix query until `OVERALL: PASS`.


## EBNF Hint: Alias Scope (`FROM ... e` vs `AS Employee`)
```ebnf
query              = "SELECT", select_list, "FROM", table_ref, [where_clause] ;

table_ref          = table_name, ["AS"], table_alias ;
select_item        = value_expr, ["AS"], output_col_alias ;
select_list        = select_item, {",", select_item} ;

value_expr         = qualified_col | unqualified_col | scalar_subquery ;
qualified_col      = table_alias, ".", col_name ;
unqualified_col    = col_name ;

where_clause       = "WHERE", predicate ;
predicate          = qualified_col, compare_op, value_expr ;
compare_op         = "=" | "<>" | ">" | "<" | ">=" | "<=" ;

table_name         = identifier ;
table_alias        = identifier ;
output_col_alias   = identifier ;
col_name           = identifier ;
identifier         = letter, {letter | digit | "_"} ;
```

Scope notes:
- `table_alias` (like `e`) is visible in that query block and used for column qualification (`e.name`).
- `output_col_alias` (like `AS Employee`) names the returned column schema only.
- These are separate namespaces; same text can be reused without conflict.


## Exercise Critique (LeetCode 181)

1. Complexity and Trade-offs of all solution attempts, with the main emphasis on the last attempt.
- Final solution is a correlated subquery comparing employee salary vs manager salary.
- Typical complexity is `O(n * manager_lookup)`; with indexed manager id lookups this is often close to linear in practice.
- Tradeoff: concise syntax, but self-join form is often easier to optimize/reason about at scale.
- Correctness: handles `NULL` manager rows naturally (comparison with `NULL` is not true), and preserves duplicate employee names when multiple rows qualify.

2. Critique of the problem-solving approach, including progression of thought and method.
- Strong: you asked the right questions about alias scope (`e`) and output aliasing (`AS Employee`).
- Strong: you validated schema expectations via harness, not just visual output.
- Gap: progression had limited independent rewrite attempts between conceptual questions and final SQL.
- Gap: could explicitly state the contract before coding: relation, predicate, output schema.

3. Improvements to Algorithm/ Optimal Example (include python solution code here in ``` ``` grouping braces)
```python
from typing import List, Optional, Tuple, Dict

def employees_more_than_manager(rows: List[Tuple[int, str, int, Optional[int]]]) -> List[str]:
    # Build manager salary lookup by employee id.
    salary_by_id: Dict[int, int] = {emp_id: salary for emp_id, _, salary, _ in rows}

    out: List[str] = []
    for _, name, salary, manager_id in rows:
        if manager_id is None:
            continue
        mgr_salary = salary_by_id.get(manager_id)
        if mgr_salary is not None and salary > mgr_salary:
            out.append(name)
    return out
```
- SQL transfer: this mirrors self-join/hash-join logic (lookup then compare).

4. Applications in real-life situations, with examples from big tech and startups (frontier tech) of the exact problem I'm solving and the solution approach. Give exact examples and usages of the generalization of the solution or pattern. (Use search for examples if needed). Be critical and outline other algorithmic tradeoffs and when I'll use this algorithmic design/ data structure approach and when I should not.
- Transferable systems pattern: single-table parent-child comparison (self-referential foreign key).
- Literal vs analogy:
  - Direct: employee-manager, category-parent, account-parent checks in relational systems.
  - Conceptual: policy inheritance trees (org/folder/project) where child attributes are compared against parent policy/state.
- Concrete examples:
  - Snowflake documentation shows employee/manager self-join and hierarchical CTE patterns.
  - GitHub nested teams and Google Cloud resource hierarchy both model strict parent-child links conceptually similar to this join pattern.
- Use when: one-parent-per-row and immediate-parent comparison is required.
- Do not use when: you need deep ancestry traversal (use recursive CTE/closure table) or graph-like many-to-many traversal.

5. Open Questions to Challenge My Understanding (non-spoiler). Ask 3-6 targeted questions tied to likely blind spots from my solution and reasoning.
- What exact SQL truth value appears when `managerId` is `NULL`, and how does that affect `WHERE` filtering?
- Under what data shape would an explicit self-join be preferable to a correlated subquery?
- If two employees share the same name, should output be deduplicated or row-faithful, and why?
- If salary can be `NULL`, what behavior do you want and how will SQL three-valued logic influence it?

6. Next-Step Application Challenges (Similar but Variant) with Learning-Goal Intent. Provide 2-4 concise challenge prompts that are close to the current problem but differ in one key dimension (constraints, interface, mutability, streaming, memory, distributed setting, etc.). For each challenge include:
- Challenge: employees earning at least 20% more than manager.
  - Learning goal intent: move from boolean comparison to derived ratio threshold.
  - What changed from the original problem: strict `>` becomes percentage comparison.
  - Why this change matters for design decisions: requires robust numeric/casting and zero/NULL handling.
- Challenge: employees earning more than any ancestor manager.
  - Learning goal intent: practice recursive hierarchical querying.
  - What changed from the original problem: immediate parent becomes transitive ancestors.
  - Why this change matters for design decisions: requires recursive CTE, cycle-safe logic, and deeper complexity handling.
- Challenge: maintain violating employees under streaming salary updates.
  - Learning goal intent: translate batch SQL logic to incremental state maintenance.
  - What changed from the original problem: static snapshot becomes mutable event stream.
  - Why this change matters for design decisions: introduces update propagation, consistency, and state-index tradeoffs.
